# Clase 231 — Capstone 3: visión por computadora con transfer learning

Pipeline mínimo CPU-friendly sobre patches sintéticos 32×32 (círculo / cuadrado / triángulo). Replica el flujo completo del capstone: baseline → features → augmentation → CNN (opcional torch) → métricas + slice analysis → stubs de transfer learning (timm), export ONNX y serving FastAPI.

Requiere: `pip install numpy scikit-learn matplotlib`. Opcional: `torch torchvision timm pytorch-lightning albumentations onnx fastapi`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
SEED = 42
IMG = 32
N = 1000
CLASSES = ['circle', 'square', 'triangle']

def draw_circle(img, cx, cy, r):
    yy, xx = np.ogrid[:IMG, :IMG]
    img[(xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2] = 1.0

def draw_square(img, cx, cy, r):
    img[max(0, cy-r):cy+r, max(0, cx-r):cx+r] = 1.0

def draw_triangle(img, cx, cy, r):
    for dy in range(-r, r+1):
        width = r - abs(dy)
        y = cy + dy
        if 0 <= y < IMG:
            img[y, max(0, cx-width):cx+width+1] = 1.0

DRAWERS = [draw_circle, draw_square, draw_triangle]

def gen_dataset(n, rng):
    X = np.zeros((n, IMG, IMG), dtype=np.float32)
    y = np.zeros(n, dtype=np.int64)
    sizes = np.zeros(n, dtype=np.int64)
    for i in range(n):
        cls = int(rng.integers(0, 3))
        r = int(rng.integers(4, 12))
        cx = int(rng.integers(r+1, IMG - r - 1))
        cy = int(rng.integers(r+1, IMG - r - 1))
        DRAWERS[cls](X[i], cx, cy, r)
        X[i] += rng.normal(0, 0.15, (IMG, IMG)).astype(np.float32)
        y[i] = cls
        sizes[i] = r
    return X.clip(0, 1), y, sizes

X, y, sizes = gen_dataset(N, rng)
print('X:', X.shape, '| y dist:', np.bincount(y), '| size range:', sizes.min(), '-', sizes.max())

## 1. Visualizar 9 ejemplos

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for ax, i in zip(axes.flat, rng.choice(N, 9, replace=False)):
    ax.imshow(X[i], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{CLASSES[y[i]]} (r={sizes[i]})', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Split estratificado 70/15/15

In [ ]:
X_tr, X_tmp, y_tr, y_tmp, s_tr, s_tmp = train_test_split(X, y, sizes, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_te, y_val, y_te, s_val, s_te = train_test_split(X_tmp, y_tmp, s_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)
print(f'train={len(X_tr)} val={len(X_val)} test={len(X_te)}')

## 3. Baseline: flatten + LogisticRegression

In [ ]:
clf_flat = LogisticRegression(max_iter=500, random_state=SEED)
clf_flat.fit(X_tr.reshape(len(X_tr), -1), y_tr)
acc_flat = accuracy_score(y_te, clf_flat.predict(X_te.reshape(len(X_te), -1)))
print(f'baseline flatten + LR test_acc = {acc_flat:.3f}')

## 4. Features HOG-like manuales (gradiente + histogram of orientations)

In [ ]:
def hog_like(imgs, n_bins=8, cell=8):
    """Versión minimal de HOG: gradientes + histogram of orientations por celda."""
    gx = np.zeros_like(imgs); gy = np.zeros_like(imgs)
    gx[:, :, 1:-1] = imgs[:, :, 2:] - imgs[:, :, :-2]
    gy[:, 1:-1, :] = imgs[:, 2:, :] - imgs[:, :-2, :]
    mag = np.sqrt(gx ** 2 + gy ** 2)
    ang = (np.arctan2(gy, gx) + np.pi) / (2 * np.pi)
    bins = np.clip((ang * n_bins).astype(int), 0, n_bins - 1)
    n_cells = IMG // cell
    feats = np.zeros((len(imgs), n_cells * n_cells * n_bins), dtype=np.float32)
    for k in range(len(imgs)):
        f = np.zeros((n_cells, n_cells, n_bins))
        for cy in range(n_cells):
            for cx in range(n_cells):
                ys, xs = slice(cy*cell, (cy+1)*cell), slice(cx*cell, (cx+1)*cell)
                np.add.at(f[cy, cx], bins[k, ys, xs], mag[k, ys, xs])
        feats[k] = f.ravel()
    feats /= (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-6)
    return feats

F_tr, F_val, F_te = hog_like(X_tr), hog_like(X_val), hog_like(X_te)
clf_hog = LogisticRegression(max_iter=500, random_state=SEED).fit(F_tr, y_tr)
acc_hog = accuracy_score(y_te, clf_hog.predict(F_te))
print(f'HOG-like + LR test_acc = {acc_hog:.3f}  (baseline flatten = {acc_flat:.3f})')

## 5. Augmentation manual: flip + rotación 90/180/270 + brightness jitter

Verificamos que las transformaciones preservan la clase (label invariance) sobre las 3 formas.

In [ ]:
def augment_batch(imgs, labels, rng):
    out = imgs.copy()
    for i in range(len(out)):
        op = rng.integers(0, 5)
        if op == 0: out[i] = np.fliplr(out[i])
        elif op == 1: out[i] = np.rot90(out[i], 1)
        elif op == 2: out[i] = np.rot90(out[i], 2)
        elif op == 3: out[i] = np.rot90(out[i], 3)
        else: out[i] = np.clip(out[i] * rng.uniform(0.7, 1.3) + rng.uniform(-0.1, 0.1), 0, 1)
    return out, labels

fig, axes = plt.subplots(1, 5, figsize=(10, 2.2))
idx = 0
axes[0].imshow(X_tr[idx], cmap='gray'); axes[0].set_title(f'orig ({CLASSES[y_tr[idx]]})'); axes[0].axis('off')
for j in range(4):
    a, _ = augment_batch(X_tr[idx:idx+1], y_tr[idx:idx+1], np.random.default_rng(j))
    axes[j+1].imshow(a[0], cmap='gray'); axes[j+1].set_title(f'aug {j+1}'); axes[j+1].axis('off')
plt.tight_layout(); plt.show()

## 6. Entrenar con augmentation (×3) y comparar

In [ ]:
aug_rng = np.random.default_rng(SEED)
X_aug_list = [X_tr]; y_aug_list = [y_tr]
for _ in range(2):
    a, b = augment_batch(X_tr, y_tr, aug_rng)
    X_aug_list.append(a); y_aug_list.append(b)
X_tr_aug = np.concatenate(X_aug_list); y_tr_aug = np.concatenate(y_aug_list)
print(f'train original={len(X_tr)} | con aug={len(X_tr_aug)} (x3)')

F_tr_aug = hog_like(X_tr_aug)
clf_aug = LogisticRegression(max_iter=500, random_state=SEED).fit(F_tr_aug, y_tr_aug)
acc_aug = accuracy_score(y_te, clf_aug.predict(F_te))
print(f'HOG + LR (sin aug)  = {acc_hog:.3f}')
print(f'HOG + LR (con aug)  = {acc_aug:.3f}')

## 7. CNN mínima en PyTorch (opcional — si torch no está, saltamos)

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as Fnn
    torch.manual_seed(SEED)

    class TinyCNN(nn.Module):
        def __init__(self, n_classes=3):
            super().__init__()
            self.c1 = nn.Conv2d(1, 16, 3, padding=1)
            self.c2 = nn.Conv2d(16, 32, 3, padding=1)
            self.fc = nn.Linear(32 * 8 * 8, n_classes)
        def forward(self, x):
            x = Fnn.max_pool2d(Fnn.relu(self.c1(x)), 2)
            x = Fnn.max_pool2d(Fnn.relu(self.c2(x)), 2)
            return self.fc(x.flatten(1))

    Xt_tr = torch.from_numpy(X_tr_aug).unsqueeze(1)
    yt_tr = torch.from_numpy(y_tr_aug)
    Xt_te = torch.from_numpy(X_te).unsqueeze(1)

    model = TinyCNN()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    bs = 64
    for epoch in range(5):
        perm = torch.randperm(len(Xt_tr))
        losses = []
        for i in range(0, len(Xt_tr), bs):
            idx = perm[i:i+bs]
            logits = model(Xt_tr[idx])
            loss = Fnn.cross_entropy(logits, yt_tr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        print(f'epoch {epoch+1}: loss={np.mean(losses):.4f}')

    with torch.no_grad():
        pred_cnn = model(Xt_te).argmax(1).numpy()
    acc_cnn = accuracy_score(y_te, pred_cnn)
    print(f'\nTinyCNN test_acc = {acc_cnn:.3f}')
except ImportError:
    print('torch no disponible — saltando CNN (esperado en CI minimal). Usá HOG + LR como modelo final.')
    pred_cnn = None

## 8. Stub: transfer learning real con `timm`

Así se reemplaza el `TinyCNN` por un backbone moderno preentrenado en ImageNet. Es lo que pide el homework.

In [ ]:
TIMM_SNIPPET = '''
import timm, torch, torch.nn as nn

# 1) Backbone preentrenado en ImageNet-21k
model = timm.create_model("convnext_tiny", pretrained=True, num_classes=NUM_CLASSES)

# 2) Fase A - Feature extraction: congelar TODO menos el head
for name, p in model.named_parameters():
    p.requires_grad = ("head" in name)

# 3) Fase B - Fine-tuning progresivo con LR diferencial
for p in model.parameters(): p.requires_grad = True
backbone_params = [p for n, p in model.named_parameters() if "head" not in n]
head_params     = [p for n, p in model.named_parameters() if "head" in n]
opt = torch.optim.AdamW([
    {"params": backbone_params, "lr": 1e-5},   # pesos ImageNet: LR chico
    {"params": head_params,     "lr": 1e-3},   # head random: LR grande
], weight_decay=0.05)

# 4) Optimizaciones torch 2.x
model = torch.compile(model, mode="reduce-overhead")
# Lightning: Trainer(precision="16-mixed", accumulate_grad_batches=4, deterministic=True)
'''
print(TIMM_SNIPPET)

## 9. Métricas: per-class F1 + confusion matrix

In [ ]:
y_pred = clf_aug.predict(F_te)
print(classification_report(y_te, y_pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3), CLASSES); ax.set_yticks(range(3), CLASSES)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_title('Confusion matrix (HOG + aug)'); plt.tight_layout(); plt.show()

## 10. Slice analysis: accuracy por tamaño del objeto

Pregunta clave: ¿el modelo es peor con figuras chicas? Si lo es y desplegás esto en producción, vas a fallar sistemáticamente en una sub-población.

In [ ]:
bins = [(4, 6, 'small'), (7, 8, 'medium'), (9, 11, 'large')]
print(f'{"slice":<10} {"n":>5} {"accuracy":>10} {"f1_macro":>10}')
print('-' * 40)
for lo, hi, name in bins:
    mask = (s_te >= lo) & (s_te <= hi)
    if mask.sum() == 0: continue
    acc = accuracy_score(y_te[mask], y_pred[mask])
    f1m = f1_score(y_te[mask], y_pred[mask], average='macro')
    print(f'{name:<10} {mask.sum():>5} {acc:>10.3f} {f1m:>10.3f}')
print('\nSi "small" subperforma mucho -> upweight objetos chicos en train o aug con random crop agresivo.')

## 11. Stub: export a ONNX

In [ ]:
ONNX_SNIPPET = '''
import torch, onnx, onnxruntime as ort, numpy as np

model.eval()
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model, dummy, "model.onnx",
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

# Validar paridad torch <-> onnxruntime
sess = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])
out_onnx = sess.run(None, {"input": dummy.numpy()})[0]
with torch.no_grad(): out_torch = model(dummy).numpy()
assert np.allclose(out_torch, out_onnx, atol=1e-4), "Mismatch torch vs ONNX"
print("OK: ONNX coincide con PyTorch dentro de 1e-4")
'''
print(ONNX_SNIPPET)

## 12. Stub: serving FastAPI con endpoint `/predict`

In [ ]:
FASTAPI_SNIPPET = '''
# serve.py
import base64, io, numpy as np, onnxruntime as ort
from fastapi import FastAPI
from pydantic import BaseModel
from PIL import Image

CLASSES = ["circle", "square", "triangle"]
sess = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])
app = FastAPI(title="Capstone-3 vision serving")

class PredictIn(BaseModel):
    image_b64: str   # imagen RGB en base64

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

@app.post("/predict")
def predict(payload: PredictIn):
    img = Image.open(io.BytesIO(base64.b64decode(payload.image_b64))).convert("RGB").resize((224, 224))
    arr = (np.asarray(img, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None]
    logits = sess.run(None, {"input": arr})[0]
    probs = softmax(logits)[0]
    k = int(probs.argmax())
    return {"class": CLASSES[k], "prob": float(probs[k])}

# Lanzar:  uvicorn serve:app --host 0.0.0.0 --port 8000
# Probar:  curl -X POST localhost:8000/predict -H "Content-Type: application/json" -d @payload.json
'''
print(FASTAPI_SNIPPET)

## Checklist de entregables del capstone

- [ ] Dataset elegido + EDA (distribución de clases, resolución, ejemplos).
- [ ] Baseline tonto (CNN from scratch) reportado como piso.
- [ ] Backbone moderno vía `timm` con feature extraction + fine-tuning progresivo.
- [ ] Augmentation con Albumentations (RandAugment + MixUp + CutMix) + ablation.
- [ ] Training con Lightning + AMP (`precision="16-mixed"`) + `torch.compile` + `seed_everything(42, workers=True)`.
- [ ] Métricas: accuracy global, **per-class F1**, **confusion matrix**, **slice analysis** sobre >=1 dimensión.
- [ ] Fairness check (Clase 224) si hay atributos sensibles.
- [ ] Export a ONNX validado vs PyTorch (`np.allclose` atol=1e-4).
- [ ] `serve.py` con FastAPI `/predict` respondiendo en < 500 ms en CPU.
- [ ] README del repo con resultados, decisiones y limitaciones.

## ✅ Soluciones de los ejercicios

Capstone de visión con `torch`/`timm`/`onnxruntime` **no instalados** en este entorno: damos el
**código correcto de referencia** (CNN, transfer learning con timm, fine-tuning diferencial,
export ONNX, serving) como strings, y un **núcleo ejecutable** con `numpy`+`sklearn` sobre
imágenes sintéticas que reproduce la *lógica* de cada hito (baseline débil → features →
augmentation → parity de inferencia). Sin internet.

### Ejercicio 1 — Baseline tonto (piso de rendimiento)

La CNN de 2 conv *from scratch* es `CNN_CODE` (PyTorch). Como piso ejecutable usamos
`LogisticRegression` sobre **píxeles aplanados**: con jitter de posición, el espacio de píxeles
no es linealmente separable y la accuracy queda baja — justo el "piso" que hay que superar.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

def gen_images(n_per=140, H=20, C=4, seed=0):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for c in range(C):
        for _ in range(n_per):
            img = rng.normal(0.15, 0.14, (H, H))            # fondo ruidoso
            dx, dy = rng.integers(-5, 6), rng.integers(-5, 6)   # jitter de posicion fuerte
            amp = 0.5
            if c == 0:                       # barra horizontal
                r = np.clip(H // 2 + dx, 1, H - 2); img[r - 1:r + 1, :] += amp
            elif c == 1:                     # barra vertical
                col = np.clip(H // 2 + dy, 1, H - 2); img[:, col - 1:col + 1] += amp
            elif c == 2:                     # diagonal
                for k in range(H):
                    img[k, np.clip(k + dy, 0, H - 1)] += amp
            else:                            # blob
                cx, cy = np.clip(H // 2 + dx, 2, H - 3), np.clip(H // 2 + dy, 2, H - 3)
                img[cx - 2:cx + 2, cy - 2:cy + 2] += amp + 0.1
            X.append(np.clip(img, 0, 1)); y.append(c)
    return np.array(X), np.array(y)

X_img, y = gen_images()
Xtr_i, Xte_i, ytr, yte = train_test_split(X_img, y, test_size=0.25, random_state=0, stratify=y)
flat_tr, flat_te = Xtr_i.reshape(len(Xtr_i), -1), Xte_i.reshape(len(Xte_i), -1)
base = LogisticRegression(max_iter=2000).fit(flat_tr, ytr)
acc_base = accuracy_score(yte, base.predict(flat_te))
print("Baseline pixeles aplanados  val_acc = %.3f (piso, multiclase C=4)" % acc_base)

CNN_CODE = '''
import torch.nn as nn
class TinyCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32 * 5 * 5, n_classes))
    def forward(self, x): return self.net(x)
'''
assert 0.0 <= acc_base <= 1.0
print("OK ejercicio 1 - baseline establecido (CNN_CODE = version PyTorch)")

### Ejercicio 2 — Feature extraction

El transfer learning real es `TIMM_CODE` (backbone `convnext_tiny` congelado + head), que sobre
imágenes reales supera al baseline por amplio margen. Como equivalente ejecutable extraemos
**features HOG-like** (histograma de orientaciones del gradiente), robustas a la posición, y
entrenamos un clasificador lineal: con el jitter fuerte del dataset, **alcanzan o superan** al
baseline de píxeles.

In [ ]:
def hog_like(imgs, n_bins=8, cell=5):
    feats = []
    for img in imgs:
        gx = np.zeros_like(img); gy = np.zeros_like(img)
        gx[:, 1:-1] = img[:, 2:] - img[:, :-2]
        gy[1:-1, :] = img[2:, :] - img[:-2, :]
        mag = np.sqrt(gx ** 2 + gy ** 2)
        ang = (np.arctan2(gy, gx) + np.pi)                 # [0, 2pi)
        H = img.shape[0]; vec = []
        for i in range(0, H, cell):
            for j in range(0, H, cell):
                m = mag[i:i + cell, j:j + cell].ravel()
                a = ang[i:i + cell, j:j + cell].ravel()
                hist, _ = np.histogram(a, bins=n_bins, range=(0, 2 * np.pi), weights=m)
                vec.append(hist)
        feats.append(np.concatenate(vec))
    return np.array(feats)

F_tr, F_te = hog_like(Xtr_i), hog_like(Xte_i)
clf_hog = LogisticRegression(max_iter=3000).fit(F_tr, ytr)
acc_hog = accuracy_score(yte, clf_hog.predict(F_te))
print("Features HOG-like  val_acc = %.3f  (baseline pixeles = %.3f)" % (acc_hog, acc_base))

TIMM_CODE = '''
import timm, torch
model = timm.create_model("convnext_tiny", pretrained=True, num_classes=N)
for p in model.parameters(): p.requires_grad = False      # congelar backbone
for p in model.get_classifier().parameters(): p.requires_grad = True
# entrenar SOLO la head 5 epochs con el backbone congelado
'''
assert acc_hog >= acc_base - 0.05, "las features de orientacion son competitivas/superan al baseline"
print("OK ejercicio 2 - feature extraction (TIMM_CODE = transfer real, superior en imagenes reales)")

### Ejercicio 3 — Fine-tuning progresivo (referencia)

El descongelado por bloques con **learning rate diferencial** (backbone 1e-5, head 1e-3) es
propio de PyTorch. Lo dejamos como `FINETUNE_CODE`; el efecto conceptual (más capacidad
entrenable → mejor ajuste) lo emulamos permitiendo más features/regularización al clasificador.

In [ ]:
FINETUNE_CODE = '''
# Fase 1: solo head (5 epochs)               -> ver TIMM_CODE
# Fase 2: descongelar ultimo bloque, LR 1e-4 (5 epochs)
for p in model.stages[-1].parameters(): p.requires_grad = True
# Fase 3: full unfreeze con LR diferencial (10 epochs)
opt = torch.optim.AdamW([
    {"params": model.stem.parameters(),      "lr": 1e-5},   # backbone: LR chico
    {"params": model.get_classifier().parameters(), "lr": 1e-3},  # head: LR grande
])
'''
# Emulacion: "mas capacidad" ~ menos regularizacion en el clasificador de features
acc_by_C = {}
for C in [0.1, 1.0, 10.0]:
    m = LogisticRegression(max_iter=3000, C=C).fit(F_tr, ytr)
    acc_by_C[C] = accuracy_score(yte, m.predict(F_te))
print("Emulacion capacidad (C de LogReg):", {k: round(v, 3) for k, v in acc_by_C.items()})
best_C = max(acc_by_C, key=acc_by_C.get)
print("mejor capacidad: C=%s -> val_acc=%.3f" % (best_C, acc_by_C[best_C]))
assert acc_by_C[best_C] >= acc_hog - 0.02
print("OK ejercicio 3 - fine-tuning progresivo (FINETUNE_CODE = LR diferencial real)")

### Ejercicio 4 — Augmentation ablation

Comparamos (a) sin aug, (b) flip + crop, (c) "RandAugment-like" (flip+crop+ruido+brillo),
(d) c + MixUp. Implementamos las transformaciones a mano y reportamos la curva de `val_acc`.

In [ ]:
rng = np.random.default_rng(1)

def flip_crop(imgs):
    out = []
    for im in imgs:
        if rng.random() < 0.5: im = np.fliplr(im)
        s = rng.integers(-2, 3)
        im = np.roll(im, s, axis=rng.integers(0, 2))
        out.append(im)
    return np.array(out)

def randaug(imgs):
    out = flip_crop(imgs)
    out = out + rng.normal(0, 0.05, out.shape)             # ruido
    out = np.clip(out * rng.uniform(0.8, 1.2), 0, 1)       # brillo
    return out

def mixup(imgs, labels, C=4):
    idx = rng.permutation(len(imgs)); lam = 0.7
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    # etiqueta soft -> tomamos la dominante (lam>0.5) para clasificador duro
    return mixed, labels

def train_eval(feat_tr, y_tr):
    m = LogisticRegression(max_iter=3000).fit(feat_tr, y_tr)
    return accuracy_score(yte, m.predict(F_te))

# (a) sin aug
acc_a = train_eval(F_tr, ytr)
# (b) flip+crop  -> duplicar dataset aumentado
Xb = np.concatenate([Xtr_i, flip_crop(Xtr_i)]); yb = np.concatenate([ytr, ytr])
acc_b = train_eval(hog_like(Xb), yb)
# (c) randaug
Xc = np.concatenate([Xtr_i, randaug(Xtr_i)]); yc = np.concatenate([ytr, ytr])
acc_c = train_eval(hog_like(Xc), yc)
# (d) randaug + mixup
Xm, ym = mixup(Xtr_i, ytr)
Xd = np.concatenate([Xc, Xm]); yd = np.concatenate([yc, ym])
acc_d = train_eval(hog_like(Xd), yd)

curva = {"a_sin_aug": acc_a, "b_flip_crop": acc_b, "c_randaug": acc_c, "d_+mixup": acc_d}
for k, v in curva.items():
    print("  %-14s val_acc=%.3f" % (k, v))
assert max(acc_b, acc_c, acc_d) >= acc_a - 0.02, "la augmentation no degrada (suele ayudar)"
print("OK ejercicio 4 - ablation de augmentation ejecutada")

### Ejercicio 5 — Serving: parity de inferencia + endpoint

El export a ONNX y la validación con `onnxruntime` (`ONNX_CODE`) requieren esas libs. Como
*parity check* ejecutable, reimplementamos el forward del clasificador lineal en **numpy puro**
y verificamos que coincide con `predict_proba` a **±1e-4** (el mismo criterio del ejercicio).
El servicio es `FASTAPI_CODE` con entrada base64.

In [ ]:
import base64, io
model = clf_hog                                            # clasificador entrenado

# forward manual (lo que exportaria ONNX): softmax(X @ W.T + b)
def manual_proba(F):
    z = F @ model.coef_.T + model.intercept_
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

p_sklearn = model.predict_proba(F_te)
p_manual = manual_proba(F_te)
max_diff = np.abs(p_sklearn - p_manual).max()
print("max |sklearn - numpy| = %.2e" % max_diff)

def predict_from_b64(img_b64, H=20):
    raw = base64.b64decode(img_b64)
    img = np.frombuffer(raw, dtype=np.float64).reshape(H, H)
    feats = hog_like(img[None, ...])
    proba = manual_proba(feats)[0]
    return {"class": int(proba.argmax()), "prob": round(float(proba.max()), 4)}

demo_b64 = base64.b64encode(Xte_i[0].astype(np.float64).tobytes()).decode()
print("endpoint /predict demo ->", predict_from_b64(demo_b64))

ONNX_CODE = '''
torch.onnx.export(model, dummy_input, "model.onnx", input_names=["input"],
                  output_names=["logits"], dynamic_axes={"input": {0: "batch"}})
import onnxruntime as ort
sess = ort.InferenceSession("model.onnx")
assert abs(sess.run(None, {"input": x})[0] - torch_out).max() < 1e-4
'''
FASTAPI_CODE = '''
from fastapi import FastAPI
app = FastAPI()
@app.post("/predict")
def predict(payload: dict):     # {"image_b64": "..."}
    return predict_from_b64(payload["image_b64"])
'''
assert max_diff < 1e-4, "el forward numpy coincide con sklearn a ±1e-4 (parity ONNX)"
print("OK ejercicio 5 - parity de inferencia + endpoint base64 (ONNX_CODE = export real)")